# Практика · Детекція на практиці

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> 🔌 **Мережа не потрібна.** Усі сцени зошит малює формулами. Досить `torch`,
> `torchvision`, `numpy` і `matplotlib`.

> ⏱ Зошит навчає **54 детектори** (18 настройок × три зерна). Заміряно:
> **близько чотирьох з половиною хвилин** на чотирьох ядрах без відеокарти.
> Це нормально: тема саме про те, як одна й та сама архітектура поводиться
> при різних даних, а на це потрібні прогони.

Тема про те, що вирішує результат детектора насправді. Архітектуру ми не міняємо
жодного разу: усі 54 навчання — це той самий якірний детектор із
[теми 26](../26-one-stage/lecture.html). Міняються **дані й рішення навколо них**.

Що зробимо:

1. навчимо базовий детектор і запамʼятаємо **розкид від зерна** — одиницю виміру
   для всього далі;
2. зіпсуємо розмітку трьома способами — **зсув рамок**, **пропущені рамки**,
   **неправильний клас** — і поміряємо ціну кожного;
3. порівняємо «видалили частину **рамок**» проти «видалили стільки ж
   **зображень**»;
4. напишемо **свою аугментацію з рамками**, звіримо кожне перетворення з маскою
   через `assert` і поміряємо, що буває, коли рамку забули перетворити;
5. переберемо три політики для **рамок на межі кадру** при обрізанні;
6. змоделюємо **витік через поділ вибірки** й побачимо розмір ілюзії;
7. побудуємо сітку **порогів** і знайдемо оптимум для двох різних цін помилки;
8. порахуємо метрики, **зрозумілі замовникові**, і побачимо, як вони
   перевпорядковують ті самі моделі.

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import box_iou, nms
import matplotlib.pyplot as plt

# один потік дає і швидкість на маленьких тензорах, і повторюваність чисел:
# під кількома потоками float-суми йдуть в іншому порядку
torch.set_num_threads(1)

print("torch    ", torch.__version__)
print("numpy    ", np.__version__)
print("потоків  ", torch.get_num_threads())

## 1 · Сцени 64×64 — той самий наскрізний приклад блоку

Полотно 64 на 64 пікселі, від одного до трьох предметів трьох класів (коло, квадрат,
трикутник) радіусом 6-10 пікселів, шум зі стандартним відхиленням 0.12. Рамка кожного
предмета рахується **з його маски**, тому вона істинна за побудовою — і це важливо
саме сьогодні: щоб міряти ціну помилки розмітки, потрібна розмітка **без** помилок,
від якої відлічувати.

Функція `make_scene` повертає ще й `plan` — список параметрів фігур. Він знадобиться
в розділі про поділ вибірки, щоб зробити «серії» схожих кадрів.

In [ ]:
SIZE = 64                                  # сторона полотна в пікселях
CLASS_NAMES = ["коло", "квадрат", "трикутник"]
CLASS_COUNT = 3
GRID = 8                                   # карта ознак 8×8
STRIDE = SIZE / GRID                       # крок карти: 8 пікселів на клітинку


def shape_mask(kind, center_x, center_y, radius):
    '''Маска однієї фігури на полотні 64×64.'''
    ys, xs = np.mgrid[0:SIZE, 0:SIZE]
    if kind == 0:                                   # коло
        return (xs - center_x) ** 2 + (ys - center_y) ** 2 <= radius * radius
    if kind == 1:                                   # квадрат
        return (np.abs(xs - center_x) <= radius) & (np.abs(ys - center_y) <= radius)
    # трикутник: ширина росте згори вниз
    return ((ys - center_y + radius >= 0) & (ys - center_y <= radius)
            & (np.abs(xs - center_x) <= (ys - center_y + radius) / 2.0))


def box_from_mask(mask):
    '''Рамка з маски: край + 1 по правому й нижньому боці, як в угоді COCO.'''
    ys, xs = np.nonzero(mask)
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def make_scene(rng, jitter=0, base=None):
    '''Одна сцена: картинка, рамки з масок, мітки класів і план фігур.

    Якщо передати base — сцена повторює ту саму компоновку з дрижанням центрів
    на jitter пікселів. Так робляться «серії» схожих кадрів.
    '''
    image = np.zeros((SIZE, SIZE), np.float32)
    boxes, labels, plan = [], [], []
    source = [None] * int(rng.integers(1, 4)) if base is None else base
    for item in source:
        for _attempt in range(40):
            if item is None:
                radius = int(rng.integers(6, 11))
                center_x = int(rng.integers(radius + 1, SIZE - radius - 1))
                center_y = int(rng.integers(radius + 1, SIZE - radius - 1))
                kind = int(rng.integers(0, 3))
            else:
                kind, center_x, center_y, radius = item
                if jitter:
                    center_x = int(np.clip(center_x + rng.integers(-jitter, jitter + 1),
                                           radius + 1, SIZE - radius - 2))
                    center_y = int(np.clip(center_y + rng.integers(-jitter, jitter + 1),
                                           radius + 1, SIZE - radius - 2))
            mask = shape_mask(kind, center_x, center_y, radius)
            box = box_from_mask(mask)

            # не даємо предметам злипатись більше ніж на 45 % площі нового
            too_close = False
            for previous in boxes:
                overlap_w = max(0, min(box[2], previous[2]) - max(box[0], previous[0]))
                overlap_h = max(0, min(box[3], previous[3]) - max(box[1], previous[1]))
                if overlap_w * overlap_h > 0.45 * (box[2] - box[0]) * (box[3] - box[1]):
                    too_close = True
                    break
            if too_close and item is None:
                continue

            image[mask] = 1.0
            boxes.append(box)
            labels.append(kind)
            plan.append((kind, center_x, center_y, radius))
            break
    image = np.clip(image + rng.normal(0, 0.12, (SIZE, SIZE)).astype(np.float32), 0, 1)
    return (image, np.array(boxes, np.float32).reshape(-1, 4),
            np.array(labels, np.int64), plan)


def make_dataset(seed, count):
    '''Набір сцен без плану — саме в такому вигляді його їсть навчання.'''
    rng = np.random.default_rng(seed)
    return [make_scene(rng)[:3] for _ in range(count)]


started = time.time()
train_set = make_dataset(42, 240)
test_set = make_dataset(7, 120)

TRAIN_BOXES = sum(len(scene[1]) for scene in train_set)
TEST_BOXES = sum(len(scene[1]) for scene in test_set)

print("згенеровано за %.2f с" % (time.time() - started))
print("навчальних сцен %d, істинних рамок %d" % (len(train_set), TRAIN_BOXES))
print("перевірних сцен %d, істинних рамок %d" % (len(test_set), TEST_BOXES))
print("предметів на сцену: %.2f" % (TRAIN_BOXES / len(train_set)))

### Як виглядають сцени

Подивимось на шість перших сцен разом з істинними рамками. Далі ці самі рамки ми
почнемо псувати — корисно памʼятати, з чого починали.

In [ ]:
figure, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for index, axis in enumerate(axes):
    image, boxes, labels = train_set[index]
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    for box, label in zip(boxes, labels):
        axis.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                     fill=False, color="tab:red", linewidth=1.4))
        axis.text(box[0], box[1] - 1.5, CLASS_NAMES[label], color="tab:red", fontsize=7)
    axis.set_title("сцена %d" % index, fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 2 · Детектор: той самий, що в темі 26

Архітектуру ми **не міняємо жодного разу за весь зошит**. Це принципово: якщо в кінці
виявиться, що розкид від помилок розмітки більший за розкид між архітектурами тем 25
і 26, то це буде порівняння з чесною основою.

Голова якірна: три квадратні якорі (12, 18 і 26 пікселів) у кожній клітинці карти 8×8,
focal loss з γ = 2, smooth L1 по координатах. Тіло — три згорткові блоки.

In [ ]:
ANCHOR_SIZES = (12.0, 18.0, 26.0)          # сторони квадратних якорів у пікселях
ANCHOR_COUNT = len(ANCHOR_SIZES)


def build_anchors():
    '''Три квадратні якорі в центрі кожної клітинки карти 8×8.'''
    out = []
    for row in range(GRID):
        for col in range(GRID):
            center_x = (col + 0.5) * STRIDE
            center_y = (row + 0.5) * STRIDE
            for size in ANCHOR_SIZES:
                out.append([center_x - size / 2, center_y - size / 2,
                            center_x + size / 2, center_y + size / 2])
    return torch.tensor(out, dtype=torch.float32)


ANCHORS = build_anchors()


def anchor_targets(boxes, labels, positive_iou=0.5, negative_iou=0.4):
    '''Стан кожного якоря: 1 позитивний, 0 фон, -1 ігнорується.'''
    total = ANCHORS.shape[0]
    state = torch.zeros(total, dtype=torch.long)
    class_id = torch.full((total,), -1, dtype=torch.long)
    matched = torch.zeros(total, 4)
    if len(boxes) == 0:
        return state, class_id, matched

    truth = torch.as_tensor(np.ascontiguousarray(boxes), dtype=torch.float32)
    overlaps = box_iou(ANCHORS, truth)
    best_iou, best_truth = overlaps.max(dim=1)
    state[(best_iou >= negative_iou) & (best_iou < positive_iou)] = -1
    state[best_iou >= positive_iou] = 1

    # рятівне правило: найкращий якір кожної істини стає позитивним попри поріг
    for truth_index in range(truth.shape[0]):
        anchor_index = int(overlaps[:, truth_index].argmax().item())
        state[anchor_index] = 1
        best_truth[anchor_index] = truth_index

    positive = state == 1
    label_tensor = torch.as_tensor(np.ascontiguousarray(labels), dtype=torch.long)
    class_id[positive] = label_tensor[best_truth[positive]]
    matched[positive] = truth[best_truth[positive]]
    return state, class_id, matched


def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    '''Стійка версія: крос-ентропію рахуємо з логітів, не переходячи через p.'''
    p = torch.sigmoid(logits)
    cross_entropy = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return alpha_t * cross_entropy * (1 - p_t) ** gamma


class Body(nn.Module):
    '''Тіло мережі: 64×64 → карта ознак 8×8.'''

    def __init__(self):
        super().__init__()

        def block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                nn.BatchNorm2d(out_channels), nn.ReLU(), nn.MaxPool2d(2))

        self.net = nn.Sequential(
            block(1, 8), block(8, 16), block(16, 32),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())

    def forward(self, x):
        return self.net(x)


PRIOR = 0.01                                    # бажана ймовірність предмета на старті
BIAS_INIT = -math.log((1 - PRIOR) / PRIOR)      # зсув останнього шару, тема 26


class AnchorDetector(nn.Module):
    '''Якірна голова: клас на кожен якір і зсув від якоря до рамки.'''

    def __init__(self):
        super().__init__()
        self.body = Body()
        self.classifier = nn.Conv2d(32, ANCHOR_COUNT * CLASS_COUNT, 3, padding=1)
        self.regressor = nn.Conv2d(32, ANCHOR_COUNT * 4, 3, padding=1)
        nn.init.constant_(self.classifier.bias, BIAS_INIT)

    def forward(self, x):
        features = self.body(x)
        count = x.shape[0]
        logits = self.classifier(features).permute(0, 2, 3, 1).reshape(count, -1,
                                                                      CLASS_COUNT)
        deltas = self.regressor(features).permute(0, 2, 3, 1).reshape(count, -1, 4)
        return logits, deltas


print("параметрів у детекторі: %d" % sum(p.numel() for p in AnchorDetector().parameters()))
print("якорів на сцену:        %d" % ANCHORS.shape[0])

In [ ]:
def encode_deltas(anchors, truth):
    '''Зсув від якоря до істинної рамки: два зсуви центра й два логарифми розміру.'''
    anchor_w = anchors[:, 2] - anchors[:, 0]
    anchor_h = anchors[:, 3] - anchors[:, 1]
    anchor_cx = anchors[:, 0] + anchor_w / 2
    anchor_cy = anchors[:, 1] + anchor_h / 2
    truth_w = (truth[:, 2] - truth[:, 0]).clamp(min=1.0)
    truth_h = (truth[:, 3] - truth[:, 1]).clamp(min=1.0)
    truth_cx = truth[:, 0] + truth_w / 2
    truth_cy = truth[:, 1] + truth_h / 2
    return torch.stack([(truth_cx - anchor_cx) / anchor_w,
                        (truth_cy - anchor_cy) / anchor_h,
                        torch.log(truth_w / anchor_w),
                        torch.log(truth_h / anchor_h)], dim=1)


def decode_deltas(anchors, deltas):
    '''Зворотне перетворення: зі зсувів назад у рамку xyxy.'''
    anchor_w = anchors[:, 2] - anchors[:, 0]
    anchor_h = anchors[:, 3] - anchors[:, 1]
    anchor_cx = anchors[:, 0] + anchor_w / 2
    anchor_cy = anchors[:, 1] + anchor_h / 2
    center_x = anchor_cx + deltas[:, 0] * anchor_w
    center_y = anchor_cy + deltas[:, 1] * anchor_h
    width = anchor_w * torch.exp(deltas[:, 2].clamp(max=3.0))
    height = anchor_h * torch.exp(deltas[:, 3].clamp(max=3.0))
    return torch.stack([center_x - width / 2, center_y - height / 2,
                        center_x + width / 2, center_y + height / 2], dim=1)


def pack(dataset):
    '''Готує тензори всього набору один раз — щоб не рахувати цілі в кожній епосі.'''
    images = torch.from_numpy(
        np.stack([np.ascontiguousarray(scene[0]) for scene in dataset])[:, None])
    total = ANCHORS.shape[0]
    keep = torch.zeros(len(dataset), total, dtype=torch.bool)
    positive = torch.zeros(len(dataset), total, dtype=torch.bool)
    class_target = torch.zeros(len(dataset), total, CLASS_COUNT)
    delta_target = torch.zeros(len(dataset), total, 4)
    for index, scene in enumerate(dataset):
        state, class_id, matched = anchor_targets(scene[1], scene[2])
        keep[index] = state >= 0                       # ігноровані не входять у втрату
        is_positive = state == 1
        positive[index] = is_positive
        if is_positive.any():
            class_target[index, is_positive, class_id[is_positive]] = 1.0
            delta_target[index, is_positive] = encode_deltas(ANCHORS[is_positive],
                                                             matched[is_positive])
    return images, keep, positive, class_target, delta_target


def images_of(dataset):
    return torch.from_numpy(
        np.stack([np.ascontiguousarray(scene[0]) for scene in dataset])[:, None])


def to_truth(dataset):
    '''Істина у вигляді, зручному для метрики: тензор рамок і тензор міток.'''
    out = []
    for scene in dataset:
        out.append((torch.from_numpy(np.ascontiguousarray(scene[1])).reshape(-1, 4).float(),
                    torch.from_numpy(np.ascontiguousarray(scene[2])).reshape(-1).long()))
    return out


EPOCHS = 12
LEARNING_RATE = 3e-3


def anchor_loss(model_output, keep, positive, class_target, delta_target):
    logits, deltas = model_output
    per_element = focal_loss(logits, class_target)
    classification = (per_element * keep.unsqueeze(-1)).sum()
    if positive.any():
        regression = F.smooth_l1_loss(deltas[positive], delta_target[positive],
                                      reduction="sum")
    else:
        regression = logits.sum() * 0
    return (classification + regression) / max(1, int(positive.sum().item()))


def train(dataset, seed, packed=None):
    '''Навчає детектор з нуля. Зерно керує вагами й порядком партій.'''
    images, keep, positive, class_target, delta_target = packed or pack(dataset)
    torch.manual_seed(seed)
    model = AnchorDetector()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for _epoch in range(EPOCHS):
        order = torch.randperm(len(images))
        for start in range(0, len(images), 32):
            batch = order[start:start + 32]
            loss = anchor_loss(model(images[batch]), keep[batch], positive[batch],
                               class_target[batch], delta_target[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def predict(model, images):
    '''Вихід мережі → списки (рамки, оцінки, класи) по сценах.'''
    model.eval()
    logits, deltas = model(images)
    probabilities = torch.sigmoid(logits)
    results = []
    for index in range(images.shape[0]):
        boxes = decode_deltas(ANCHORS, deltas[index])
        score, class_id = probabilities[index].max(dim=1)
        results.append((boxes, score, class_id))
    return results


test_images = images_of(test_set)
test_truth = to_truth(test_set)
print("готово: одне навчання — %d епох по %d сцен, AdamW, lr = %g"
      % (EPOCHS, len(train_set), LEARNING_RATE))

### Метрика: mAP@0.5 — та сама, що в темі 23

Зіставляємо жадібно за спаданням упевненості, вважаємо влучанням `IoU ≥ 0.5`,
рахуємо площу під огинальною кривої точність-повнота й усереднюємо по трьох класах.

In [ ]:
def greedy_match(boxes, scores, truth_boxes, iou_threshold):
    '''Жадібне зіставлення за спаданням оцінки — те саме правило, що в темі 23.'''
    order = torch.argsort(scores, descending=True)
    hits = torch.zeros(len(order))
    if len(truth_boxes) and len(order):
        overlaps = box_iou(boxes[order], truth_boxes).numpy()
        taken = np.zeros(len(truth_boxes), bool)
        # позиція, що ніде не дотягує до порога, влучити не може — не перебираємо її
        candidates = np.nonzero(overlaps.max(axis=1) >= iou_threshold)[0]
        for position in candidates:
            row = overlaps[position].copy()
            row[taken] = -1.0                     # зайняті істини більше не пропонуємо
            best = int(row.argmax())
            if row[best] >= iou_threshold:
                taken[best] = True
                hits[position] = 1.0
    return scores[order], hits


def average_precision(scores, hits, truth_count):
    '''AP як площа під огинальною кривої точність-повнота.'''
    order = torch.argsort(scores, descending=True)
    ordered_hits = hits[order].numpy()
    running_hits = np.cumsum(ordered_hits)
    running_misses = np.cumsum(1 - ordered_hits)
    precision = running_hits / np.maximum(running_hits + running_misses, 1e-9)
    recall = running_hits / truth_count
    envelope = np.maximum.accumulate(precision[::-1])[::-1]
    steps = np.diff(np.concatenate([[0.0], recall]))
    return float(np.sum(steps * envelope))


def mean_average_precision(predictions, ground_truth, iou_threshold=0.5,
                           nms_threshold=0.5, score_floor=1e-3):
    '''mAP при одному порозі IoU: середнє AP по трьох класах.'''
    values = []
    for class_index in range(CLASS_COUNT):
        score_parts, hit_parts, truth_count = [], [], 0
        for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                        ground_truth):
            chosen = (scores >= score_floor) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            truth_count += len(this_truth)
            ordered, hits = greedy_match(picked_boxes, picked_scores, this_truth,
                                         iou_threshold)
            score_parts.append(ordered)
            hit_parts.append(hits)
        values.append(average_precision(torch.cat(score_parts), torch.cat(hit_parts),
                                        max(1, truth_count)))
    return float(np.mean(values))


SEEDS = (0, 1, 2)
SCORES = {}          # назва настройки → три значення mAP
MODELS = {}          # назва настройки → модель зерна 0, для розділів 8 і 9
ALL_MODELS = {}      # назва настройки → всі три моделі, для розділу 4


def run(name, dataset, keep_model=False, keep_all=False, truth=None, images=None):
    '''Навчає три моделі на цих даних і друкує mAP по зернах.'''
    packed = pack(dataset)
    values, trained = [], []
    started = time.time()
    for seed in SEEDS:
        model = train(dataset, seed, packed=packed)
        value = mean_average_precision(predict(model, test_images if images is None
                                               else images),
                                       test_truth if truth is None else truth)
        values.append(value)
        trained.append(model)
        if seed == 0 and (keep_model or keep_all):
            MODELS[name] = model
    if keep_all:
        ALL_MODELS[name] = trained
    SCORES[name] = values
    print("  %-12s  %.4f  %.4f  %.4f   середнє %.4f   розкид %.4f   (%.0f с)"
          % (name, values[0], values[1], values[2], float(np.mean(values)),
             max(values) - min(values), time.time() - started))
    return values


print("  настройка       зерно0  зерно1  зерно2")
run("чиста", train_set, keep_model=True, keep_all=True)
SEED_SPREAD = max(SCORES["чиста"]) - min(SCORES["чиста"])
print()
print("Розкид від зерна на чистій розмітці: %.4f." % SEED_SPREAD)
print("Це одиниця виміру для всього зошита: різниця, менша за неї, нічого не доводить.")

## 3 · Скільки коштує помилка розмітки

Головний замір теми. Псуємо **тільки навчальну** розмітку — перевірна лишається
ідеальною. Три способи, кожен відповідає реальній помилці розмітника:

- **зсув рамок** на 2 і 4 пікселі — рамку вели поспіхом, вона трохи не там;
- **пропущені рамки** (5, 15 і 30 %) — предмет на картинці є, а рамки в файлі немає;
- **неправильний клас** (15 і 30 %) — рамка правильна, підпис ні.

Спершу подивимось, наскільки зсунута рамка ще схожа на істинну. `IoU` тут — та сама
міра збігу, що в [темі 23](../23-detection-setup/lecture.html).

In [ ]:
def corrupt_shift(dataset, pixels, seed=101):
    '''Зсуває кожну рамку рівно на pixels пікселів у випадковому напрямку.'''
    rng = np.random.default_rng(seed)
    out = []
    for image, boxes, labels in dataset:
        moved = boxes.copy()
        for index in range(len(moved)):
            axis = int(rng.integers(0, 2))                 # 0 — по x, 1 — по y
            step = pixels if rng.random() < 0.5 else -pixels
            if axis == 0:
                moved[index, [0, 2]] += step
            else:
                moved[index, [1, 3]] += step
        out.append((image, moved, labels))
    return out


def corrupt_miss(dataset, share, seed=202):
    '''Викидає частку рамок. Предмет на картинці лишається — зникає лише підпис.'''
    rng = np.random.default_rng(seed)
    out = []
    for image, boxes, labels in dataset:
        alive = rng.random(len(boxes)) >= share
        out.append((image, boxes[alive], labels[alive]))
    return out


def corrupt_class(dataset, share, seed=303):
    '''Міняє клас частини рамок на будь-який інший. Координати не чіпаємо.'''
    rng = np.random.default_rng(seed)
    out = []
    for image, boxes, labels in dataset:
        spoiled = labels.copy()
        for index in range(len(spoiled)):
            if rng.random() < share:
                others = [c for c in range(CLASS_COUNT) if c != labels[index]]
                spoiled[index] = others[int(rng.integers(0, len(others)))]
        out.append((image, boxes, spoiled))
    return out


print("наскільки зсунута рамка ще схожа на істинну:")
for pixels in (2, 4):
    spoiled = corrupt_shift(train_set, pixels)
    overlaps = []
    for (_image, good, _labels), (_image2, bad, _labels2) in zip(train_set, spoiled):
        if len(good):
            overlaps.append(float(box_iou(torch.from_numpy(np.ascontiguousarray(good)),
                                          torch.from_numpy(np.ascontiguousarray(bad)))
                                  .diagonal().mean()))
    print("  зсув %d px → середній IoU з істинною рамкою %.4f" % (pixels, np.mean(overlaps)))
print()
print("Нагадування з теми 23: рамка зі стороною 40 px тримає IoU 0.9 лише при зсуві")
print("до 2.1 px. Наші фігури менші, тому 4 пікселі коштують іще дорожче.")

Тепер сам замір. Вісім настройок, у кожній три зерна — це 24 навчання, найдовший
шматок зошита.

In [ ]:
print("  настройка       зерно0  зерно1  зерно2")
for pixels in (2, 4):
    run("зсув %d px" % pixels, corrupt_shift(train_set, pixels),
        keep_model=(pixels == 4))
for share in (0.05, 0.15, 0.30):
    spoiled = corrupt_miss(train_set, share)
    left = sum(len(scene[1]) for scene in spoiled)
    run("пропуск %d %%" % int(share * 100), spoiled,
        keep_model=(share == 0.15), keep_all=(share in (0.15, 0.30)))
    print("                 (рамок лишилось %d із %d)" % (left, TRAIN_BOXES))
for share in (0.15, 0.30):
    run("клас %d %%" % int(share * 100), corrupt_class(train_set, share),
        keep_model=(share == 0.30))

In [ ]:
base = float(np.mean(SCORES["чиста"]))
print("настройка         середній mAP    втрата    у скільки разів більша за розкид")
print("-" * 78)
for name in ("чиста", "зсув 2 px", "зсув 4 px", "пропуск 5 %", "пропуск 15 %",
             "пропуск 30 %", "клас 15 %", "клас 30 %"):
    value = float(np.mean(SCORES[name]))
    drop = base - value
    print("  %-14s   %.4f       %+.4f        %s"
          % (name, value, -drop,
             "—" if name == "чиста" else "%.1f×" % (drop / SEED_SPREAD)))

print()
print("Для порівняння — найбільші виграші від зміни АРХІТЕКТУРИ в цьому блоці:")
print("  тема 26, focal loss γ = 2 проти звичайної крос-ентропії:  +0.1061")
print("  тема 26, anchor-free голова проти якірної:                +0.0577")
print()
worst = base - float(np.mean(SCORES["зсув 4 px"]))
print("Зсув рамок на 4 пікселі коштує %.4f — це більше за обидва числа вище." % worst)
print("Тобто акуратність розмітника вирішує більше, ніж вибір між двома")
print("архітектурами, на які пішли роки досліджень.")

## 4 · Пропущена рамка гірша за відсутнє зображення?

Пропущена істинна рамка робить справжній предмет **негативним прикладом**: на картинці
коло є, у файлі його немає, і мережу штрафують саме за те, що вона його побачила.
Просто менша вибірка так не робить — вона нічого не стверджує неправильно.

Перевіримо це прямо: порівняємо «видалили 15 % **рамок**» проти «видалили 15 %
**зображень**», і те саме на 30 %. Кількість рамок, що лишились, у цих парах близька,
тож порівняння чесне.

In [ ]:
def drop_images(dataset, share, seed=404):
    '''Викидає частку сцен цілком — разом з картинками й рамками.'''
    rng = np.random.default_rng(seed)
    keep = rng.random(len(dataset)) >= share
    return [scene for scene, alive in zip(dataset, keep) if alive]


print("  настройка       зерно0  зерно1  зерно2")
for share in (0.15, 0.30):
    smaller = drop_images(train_set, share)
    run("−%d %% сцен" % int(share * 100), smaller, keep_all=True)
    print("                 (сцен %d, рамок %d)"
          % (len(smaller), sum(len(scene[1]) for scene in smaller)))

In [ ]:
print("частка   видалили рамки   видалили сцени   різниця")
print("-" * 56)
for share in (15, 30):
    a = float(np.mean(SCORES["пропуск %d %%" % share]))
    b = float(np.mean(SCORES["−%d %% сцен" % share]))
    print("  %2d %%      %.4f          %.4f        %+.4f" % (share, a, b, b - a))
print()
print("Розкид від зерна — %.4f." % SEED_SPREAD)
print()
print("Несподіванка: за mAP різниці майже немає, і на тридцяти відсотках два")
print("числа збігаються з точністю до третього знака. Якби ми дивились тільки")
print("на mAP, довелося б")
print("сказати «різниці не впіймано» — і це була б неправда.")
print("Подивимось на те саме інакше: не в середньому по всіх порогах, а на одному")
print("робочому порозі — і не на площу під кривою, а на впевненість, яку модель")
print("дає справжнім предметам.")

In [ ]:
def confidence_on_truth(model, iou_threshold=0.5):
    '''Яку впевненість модель дає кожному справжньому предмету перевірної вибірки.'''
    predictions = predict(model, test_images)
    best_scores = []
    for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                    test_truth):
        if len(truth_boxes) == 0:
            continue
        overlaps = box_iou(truth_boxes, boxes).numpy()
        for truth_index in range(len(truth_boxes)):
            same_class = (labels == truth_labels[truth_index]).numpy()
            good = (overlaps[truth_index] >= iou_threshold) & same_class
            best_scores.append(float(scores.numpy()[good].max()) if good.any() else 0.0)
    values = np.array(best_scores)
    return values.mean(), (values >= 0.30).mean()


# міряємо ВСІ три зерна кожної настройки: на одному зерні тут можна дістати
# протилежний висновок, і нижче ми це побачимо на власні очі
confidence_table = {}
NAMES = ("чиста", "пропуск 15 %", "−15 % сцен", "пропуск 30 %", "−30 % сцен")
for name in NAMES:
    per_seed = [confidence_on_truth(model) for model in ALL_MODELS[name]]
    confidence_table[name] = dict(
        conf=[value[0] for value in per_seed],
        rec=[value[1] for value in per_seed])

print("настройка        mAP@0.5   впевненість на істині   знайдено при порозі 0.30")
print("-" * 76)
for name in NAMES:
    row = confidence_table[name]
    print("  %-14s  %.4f          %.4f                  %.4f"
          % (name, float(np.mean(SCORES[name])),
             float(np.mean(row["conf"])), float(np.mean(row["rec"]))))

print()
print("те саме по зернах — щоб було видно, наскільки твердженню можна вірити:")
for name in NAMES:
    row = confidence_table[name]
    print("  %-14s  впевненість %s   повнота %s"
          % (name,
             " ".join("%.4f" % value for value in row["conf"]),
             " ".join("%.4f" % value for value in row["rec"])))

In [ ]:
miss_conf = confidence_table["пропуск 30 %"]["conf"]
drop_conf = confidence_table["−30 % сцен"]["conf"]
miss_rec = confidence_table["пропуск 30 %"]["rec"]
drop_rec = confidence_table["−30 % сцен"]["rec"]

print("Тридцять відсотків: mAP однаковий — %.4f проти %.4f."
      % (float(np.mean(SCORES["пропуск 30 %"])), float(np.mean(SCORES["−30 % сцен"]))))
print()
print("Але впевненість, яку модель дає справжньому предмету, — ні:")
print("  видалили рамки: %s" % " ".join("%.4f" % v for v in miss_conf))
print("  видалили сцени: %s" % " ".join("%.4f" % v for v in drop_conf))
if max(miss_conf) < min(drop_conf):
    print("  → УСІ три зерна «видалили рамки» нижчі за УСІ три зерна «видалили сцени»")
    print("    (%.4f проти %.4f). Купи зерен не перетинаються — це найсильніше"
          % (max(miss_conf), min(drop_conf)))
    print("    твердження, яке взагалі можна зробити на трьох прогонах.")
else:
    print("  → купи зерен перетинаються: різницю не впіймано")
print()
print("Повнота на робочому порозі 0.30 у середньому %.4f проти %.4f,"
      % (float(np.mean(miss_rec)), float(np.mean(drop_rec))))
print("але читай це обережно: розкид усередині «видалили рамки» дорівнює %.4f"
      % (max(miss_rec) - min(miss_rec)))
print("проти %.4f у «видалили сцени». Одне зерно там просто розвалилось"
      % (max(drop_rec) - min(drop_rec)))
print("(%.4f), і саме воно тягне середнє вниз." % min(miss_rec))
print()
print("Отже, чесне формулювання таке. Пропущена рамка НЕ забирає приклад —")
print("вона наказує мережі мовчати там, де та правильно побачила предмет.")
print("Це видно у впевненості, де всі зерна впорядковані однаково. А от у скільки")
print("разів це коштує повноти — на 240 сценах ми не зміряли: розкид завеликий.")
print()
print("І окремий урок про саму метрику: mAP цього не показав, бо він усереднює")
print("по ВСІХ порогах упевненості й тому майже не помічає, що модель стала")
print("тихішою. Замовник помітить це першого ж дня. Про це — розділ 9.")

## 5 · Аугментація з рамками

Аугментацію ми проходили в [темі 10](../10-augmentation/lecture.html), але там задачею
була класифікація: перетворив картинку — мітка лишилась тією самою. У детекції мітка
**геометрична**, і її треба перетворювати разом із зображенням. Помилитись тут легко,
а помітити помилку майже неможливо: код не падає, картинки виглядають нормально,
модель просто гірша.

Тому кожне перетворення ми пишемо парою «зображення + рамки» і **звіряємо з маскою**.
Ідея звірки проста: у нас є маска фігури; перетворимо саму маску тим самим
перетворенням і порахуємо рамку з неї заново. Якщо формула правильна, два числа
збігаються точно.

In [ ]:
def flip_h(image, boxes):
    '''Горизонтальне дзеркало. x1 і x2 міняються місцями відносно ширини.'''
    out = np.ascontiguousarray(image[:, ::-1])
    if len(boxes) == 0:
        return out, boxes
    moved = boxes.copy()
    moved[:, 0] = SIZE - boxes[:, 2]
    moved[:, 2] = SIZE - boxes[:, 0]
    return out, moved


def flip_v(image, boxes):
    '''Вертикальне дзеркало. Симетрично до горизонтального, тільки по y.'''
    out = np.ascontiguousarray(image[::-1, :])
    if len(boxes) == 0:
        return out, boxes
    moved = boxes.copy()
    moved[:, 1] = SIZE - boxes[:, 3]
    moved[:, 3] = SIZE - boxes[:, 1]
    return out, moved


def shift(image, boxes, dx, dy):
    '''Зсув на цілу кількість пікселів; порожнє місце лишається чорним.'''
    out = np.zeros_like(image)
    xs0, xs1 = max(0, dx), min(SIZE, SIZE + dx)
    ys0, ys1 = max(0, dy), min(SIZE, SIZE + dy)
    out[ys0:ys1, xs0:xs1] = image[ys0 - dy:ys1 - dy, xs0 - dx:xs1 - dx]
    if len(boxes) == 0:
        return out, boxes
    return out, boxes + np.array([dx, dy, dx, dy], np.float32)


def crop_resize(image, boxes, x0, y0, side):
    '''Вирізає вікно side×side і розтягує його назад до 64×64 — тобто наближає.'''
    window = image[y0:y0 + side, x0:x0 + side]
    tensor = torch.from_numpy(np.ascontiguousarray(window))[None, None]
    out = F.interpolate(tensor, size=(SIZE, SIZE), mode="bilinear",
                        align_corners=False)[0, 0].numpy()
    if len(boxes) == 0:
        return out, boxes
    scale = SIZE / side
    moved = boxes.copy()
    moved[:, [0, 2]] = (boxes[:, [0, 2]] - x0) * scale
    moved[:, [1, 3]] = (boxes[:, [1, 3]] - y0) * scale
    return out, moved


# ── звірка через маску ────────────────────────────────────────────────
check_rng = np.random.default_rng(5)
image, boxes, labels, plan = make_scene(check_rng)
kind, cx, cy, radius = plan[0]
mask = shape_mask(kind, cx, cy, radius)
one_box = boxes[:1]

by_formula = flip_h(image, one_box)[1][0]
by_mask = box_from_mask(np.ascontiguousarray(mask[:, ::-1]))
print("дзеркало по горизонталі: формула %s, з маски %s" % (list(by_formula), by_mask))
assert np.allclose(by_formula, by_mask), "рамка після дзеркала розійшлась із маскою!"

by_formula = flip_v(image, one_box)[1][0]
by_mask = box_from_mask(np.ascontiguousarray(mask[::-1, :]))
assert np.allclose(by_formula, by_mask), "рамка після вертикального дзеркала розійшлась!"

by_formula = shift(image, one_box, 5, -3)[1][0]
by_mask = box_from_mask(shift(mask.astype(np.float32), one_box, 5, -3)[0] > 0.5)
print("зсув на (5, −3):         формула %s, з маски %s" % (list(by_formula), by_mask))
assert np.allclose(by_formula, by_mask), "рамка після зсуву розійшлась із маскою!"

print()
print("✅ усі три перетворення дають рамку, що збігається з рамкою з маски")

### Що буває, коли рамку забули перетворити

Тепер сам замір. Щоб аугментації було де показати себе, беремо **маленьку** базу —
80 сцен. Порівнюємо три набори однакового обсягу для двох останніх:

- **без аугментації** — самі 80 сцен;
- **аугментація з рамками** — 80 сцен плюс два аугментовані проходи, разом 240;
- **аугментація без рамок** — те саме, але координати рамок лишились від оригіналу.

Друге й третє відрізняються **єдиним** рядком коду.

In [ ]:
def apply_policy(boxes, labels, policy):
    '''Що робити з рамкою, яка вилізла за кадр: drop / clip / keep.'''
    if len(boxes) == 0:
        return boxes, labels
    inside = boxes.copy()
    inside[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, SIZE)
    inside[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, SIZE)
    width = inside[:, 2] - inside[:, 0]
    height = inside[:, 3] - inside[:, 1]
    if policy == "drop":                       # викидаємо все, що торкнулось межі
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        visible = np.maximum(width, 0) * np.maximum(height, 0)
        alive = visible >= 0.999 * np.maximum(area, 1e-6)
        return boxes[alive], labels[alive]
    if policy == "clip":                       # обрізаємо рамку по краю кадру
        alive = (width >= 2) & (height >= 2)
        return inside[alive], labels[alive]
    alive = (width >= 2) & (height >= 2)       # лишаємо як є, разом із хвостом за кадром
    return boxes[alive], labels[alive]


def augment(dataset, seed, with_boxes=True):
    '''Один аугментований прохід по набору. with_boxes=False — та сама помилка.'''
    rng = np.random.default_rng(seed)
    out = []
    for image, boxes, labels in dataset:
        work, work_boxes = image, boxes.copy()
        if rng.random() < 0.5:
            work, moved = flip_h(work, work_boxes)
            work_boxes = moved if with_boxes else work_boxes
        dx = int(rng.integers(-6, 7))
        dy = int(rng.integers(-6, 7))
        work, moved = shift(work, work_boxes, dx, dy)
        work_boxes = moved if with_boxes else work_boxes
        if rng.random() < 0.5:
            side = int(rng.choice([52, 58]))
            corner = (SIZE - side) // 2
            work, moved = crop_resize(work, work_boxes, corner, corner, side)
            work_boxes = moved if with_boxes else work_boxes
        work_boxes, work_labels = apply_policy(work_boxes, labels, "clip")
        out.append((work, work_boxes, work_labels))
    return out


small_set = make_dataset(42, 80)
print("база для цього розділу: %d сцен, %d рамок"
      % (len(small_set), sum(len(scene[1]) for scene in small_set)))
print()
print("  настройка       зерно0  зерно1  зерно2")
run("без аугм.", small_set)
run("аугм. з рамк.", small_set + augment(small_set, 11) + augment(small_set, 12),
    keep_model=True)
run("аугм. без рамок", small_set + augment(small_set, 11, with_boxes=False)
    + augment(small_set, 12, with_boxes=False), keep_model=True)

In [ ]:
none_value = float(np.mean(SCORES["без аугм."]))
ok_value = float(np.mean(SCORES["аугм. з рамк."]))
bad_value = float(np.mean(SCORES["аугм. без рамок"]))

print("без аугментації (80 сцен):        %.4f" % none_value)
print("аугментація з рамками (240):      %.4f   (%+.4f)" % (ok_value, ok_value - none_value))
print("аугментація без рамок (240):      %.4f   (%+.4f)" % (bad_value, bad_value - none_value))
print()
print("Ціна забутої строчки: %.4f — це %.0f %% усього виграшу від аугментації."
      % (ok_value - bad_value, 100 * (ok_value - bad_value) / (ok_value - none_value)))
print()
print("Зверни увагу: варіант «без рамок» усе одно кращий за «без аугментації».")
print("Саме тому помилку так важко помітити — вона не ламає навчання, а тихо")
print("зʼїдає більшу частину користі. Метрика росте, ти радієш і йдеш далі.")

## 6 · Рамка на межі кадру

Обрізання (crop) — найкорисніша аугментація для детекції: воно міняє і положення, і
масштаб предмета. Але воно ставить питання, на яке немає очевидної відповіді: що
робити з рамкою, яка після обрізання наполовину вийшла за кадр?

Три політики, і кожна десь використовується:

- **викинути** — рамка, що торкнулась межі, зникає. Предмет на картинці лишається;
- **обрізати** — рамка обрізається по краю кадру; підпис описує видиму частину;
- **лишити** — рамка зберігає початкові координати, частина яких за кадром.

Перша політика робить те саме, що пропущена рамка з розділу 4: предмет видно, підпису
немає. Третя вчить регресор передбачати координати, яких на картинці не видно.

In [ ]:
def random_crop(dataset, seed, policy, side=48, probability=0.5):
    '''Обрізає половину сцен вікном side×side і застосовує обрану політику.'''
    rng = np.random.default_rng(seed)
    out, at_border = [], 0
    for image, boxes, labels in dataset:
        if rng.random() >= probability:
            out.append((image, boxes, labels))
            continue
        x0 = int(rng.integers(0, SIZE - side + 1))
        y0 = int(rng.integers(0, SIZE - side + 1))
        work, moved = crop_resize(image, boxes, x0, y0, side)
        for box in moved:
            if box[0] < 0 or box[1] < 0 or box[2] > SIZE or box[3] > SIZE:
                at_border += 1
        kept_boxes, kept_labels = apply_policy(moved, labels, policy)
        out.append((work, kept_boxes, kept_labels))
    return out, at_border


print("  настройка       зерно0  зерно1  зерно2")
for policy, title in (("drop", "викинути"), ("clip", "обрізати"), ("keep", "лишити")):
    cropped, at_border = random_crop(train_set, 77, policy)
    left = sum(len(scene[1]) for scene in cropped)
    run("межа: %s" % title, cropped)
    print("                 (рамок %d із %d, на межі побувало %d)"
          % (left, TRAIN_BOXES, at_border))

## 7 · Витік через поділ вибірки

Найтихіша помилка з усіх. Якщо кадри одного відео потрапили і в навчання, і в
перевірку, модель на перевірці впізнає **майже ті самі картинки**, які бачила під час
навчання. Метрика виходить завищеною, і ти дізнаєшся про це вже на реальних даних.

Змоделюймо це. Зробимо 60 «серій» по 4 схожі кадри: та сама компоновка предметів,
центри дрижать на ±1 піксель, шум щоразу новий. Це модель кадрів з одного відео.

Поділимо 240 кадрів на 180 навчальних і 60 перевірних двома способами — випадково
й **за серіями** — і поміряємо кожну модель двічі: на своїй перевірці й на цілком
незалежному наборі з початку зошита.

In [ ]:
def make_series_dataset(seed, series_count, per_series, jitter=1):
    '''series_count серій по per_series схожих кадрів. Повертає набір і номери серій.'''
    rng = np.random.default_rng(seed)
    scenes, group = [], []
    for series_index in range(series_count):
        image, boxes, labels, plan = make_scene(rng)
        scenes.append((image, boxes, labels))
        group.append(series_index)
        for _copy in range(per_series - 1):
            image2, boxes2, labels2, _plan = make_scene(rng, jitter=jitter, base=plan)
            scenes.append((image2, boxes2, labels2))
            group.append(series_index)
    return scenes, np.array(group)


series_set, series_group = make_series_dataset(2024, 60, 4)
order = np.random.default_rng(9).permutation(len(series_set))
random_train, random_val = order[:180], order[180:]
group_train = np.nonzero(series_group < 45)[0]
group_val = np.nonzero(series_group >= 45)[0]

shared = len(set(series_group[random_train]) & set(series_group[random_val]))
print("серій %d, кадрів %d" % (len(set(series_group)), len(series_set)))
print("випадковий поділ: серій, представлених і в навчанні, і в перевірці: %d із 60" % shared)
print("поділ за серіями: таких серій %d"
      % len(set(series_group[group_train]) & set(series_group[group_val])))

In [ ]:
leak = {}
for title, train_index, val_index in (("випадковий", random_train, random_val),
                                      ("за серіями", group_train, group_val)):
    subset = [series_set[i] for i in train_index]
    validation = [series_set[i] for i in val_index]
    val_images, val_truth = images_of(validation), to_truth(validation)
    packed = pack(subset)
    own, outside = [], []
    for seed in SEEDS:
        model = train(subset, seed, packed=packed)
        own.append(mean_average_precision(predict(model, val_images), val_truth))
        outside.append(mean_average_precision(predict(model, test_images), test_truth))
    leak[title] = (float(np.mean(own)), float(np.mean(outside)))
    print("%-11s поділ: своя перевірка %.4f | незалежний набір %.4f | ілюзія %+.4f"
          % (title, np.mean(own), np.mean(outside), np.mean(own) - np.mean(outside)))

print()
print("Головне — у другій колонці. Обидві моделі однаково добрі насправді")
print("(%.4f і %.4f), і вся різниця в перших колонках — це ілюзія."
      % (leak["випадковий"][1], leak["за серіями"][1]))
print("Випадковий поділ обіцяє %.4f, чесна оцінка — %.4f."
      % (leak["випадковий"][0], leak["випадковий"][1]))

## 8 · Пороги — це рішення, а не константи

Поріг упевненості й поріг NMS зазвичай беруть із чужого коду: 0.5 і 0.5. Це не
константи природи, а **два налаштування, які треба обрати під ціну помилки**.

Побудуємо сітку: 9 значень порога впевненості × 6 значень порога NMS. У кожній
клітинці порахуємо три числа — влучання (TP), хибні спрацювання (FP) і пропуски (FN).
Потім призначимо помилкам ціну й знайдемо, де вартість найменша.

In [ ]:
CONFIDENCE_GRID = [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
NMS_GRID = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]

raw_predictions = predict(MODELS["чиста"], test_images)


def count_errors(confidence_threshold, nms_threshold, iou_threshold=0.5):
    '''Влучання, хибні спрацювання й пропуски на всій перевірній вибірці.'''
    true_positive = false_positive = false_negative = 0
    for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(raw_predictions,
                                                                    test_truth):
        for class_index in range(CLASS_COUNT):
            chosen = (scores >= confidence_threshold) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            _ordered, hits = greedy_match(picked_boxes, picked_scores, this_truth,
                                          iou_threshold)
            hit = int(hits.sum())
            true_positive += hit
            false_positive += len(picked_boxes) - hit
            false_negative += len(this_truth) - hit
    return true_positive, false_positive, false_negative


started = time.time()
grid = [[count_errors(c, n) for n in NMS_GRID] for c in CONFIDENCE_GRID]
print("сітку %d × %d порахували за %.1f с"
      % (len(CONFIDENCE_GRID), len(NMS_GRID), time.time() - started))

In [ ]:
FRAMES = len(test_set)


def best_thresholds(cost_miss, cost_false):
    '''Пара порогів із найменшою очікуваною вартістю на кадр.'''
    best = None
    for i, confidence in enumerate(CONFIDENCE_GRID):
        for j, nms_threshold in enumerate(NMS_GRID):
            tp, fp, fn = grid[i][j]
            cost = (cost_miss * fn + cost_false * fp) / FRAMES
            if best is None or cost < best[0]:
                best = (cost, confidence, nms_threshold, tp, fp, fn)
    return best


settings = {}
print("ціна помилки                 упевн.  NMS   вартість/кадр    TP    FP    FN")
print("-" * 78)
for cost_miss, cost_false, title in ((10, 1, "пропуск дорогий  (10 : 1)"),
                                     (1, 1, "однаково         ( 1 : 1)"),
                                     (1, 10, "хибне дороге     ( 1 : 10)")):
    cost, confidence, nms_threshold, tp, fp, fn = best_thresholds(cost_miss, cost_false)
    settings[(cost_miss, cost_false)] = (confidence, nms_threshold, cost)
    print("  %-26s %.2f   %.2f     %8.3f    %4d  %4d  %4d"
          % (title, confidence, nms_threshold, cost, tp, fp, fn))

In [ ]:
def cost_at(confidence, nms_threshold, cost_miss, cost_false):
    i = CONFIDENCE_GRID.index(confidence)
    j = NMS_GRID.index(nms_threshold)
    tp, fp, fn = grid[i][j]
    return (cost_miss * fn + cost_false * fp) / FRAMES


miss_conf, miss_nms, miss_cost = settings[(10, 1)]
false_conf, false_nms, false_cost = settings[(1, 10)]

wrong_here = cost_at(miss_conf, miss_nms, 1, 10)
wrong_there = cost_at(false_conf, false_nms, 10, 1)

print("Що коштує взяти чужі пороги:")
print()
print("  пороги «дорогий пропуск» у світі «дорогі хибні»:  %.3f проти %.3f — у %.1f раза гірше"
      % (wrong_here, false_cost, wrong_here / false_cost))
print("  пороги «дорогі хибні» у світі «дорогий пропуск»:  %.3f проти %.3f — у %.1f раза гірше"
      % (wrong_there, miss_cost, wrong_there / miss_cost))
print()
print("І окремо — знамените «0.5 і 0.5», яке стоїть у половині чужого коду:")
for cost_miss, cost_false, title in ((10, 1, "пропуск дорогий"), (1, 10, "хибне дороге")):
    default_cost = cost_at(0.55, 0.50, cost_miss, cost_false)
    best_cost = settings[(cost_miss, cost_false)][2]
    print("  %-16s  0.55 / 0.50 дає %.3f, оптимум %.3f — у %.1f раза гірше"
          % (title, default_cost, best_cost, default_cost / best_cost))

## 9 · Метрика, зрозуміла замовникові

`mAP` — правильна метрика для порівняння моделей і **погана** для розмови з людиною,
яка не інженер. Вона усереднює по класах і по порогах упевненості, тобто описує
модель у всіх режимах одразу, включно з тими, у яких її ніхто не запускатиме.

Порахуємо натомість два числа, зміст яких зрозумілий без пояснень:

- **частка кадрів, де знайдено всі предмети** — «на скількох знімках зі ста нічого
  не загубилось»;
- **середня кількість хибних спрацювань на кадр** — «скільки зайвого доведеться
  відкидати руками».

Обидва рахуються при **одному** робочому порозі, а не в середньому по всіх.

In [ ]:
def client_metrics(model, confidence_threshold=0.30, nms_threshold=0.50,
                   iou_threshold=0.5):
    '''Метрики для розмови із замовником — при одному робочому порозі.'''
    predictions = predict(model, test_images)
    full_frames = false_total = found = truth_total = 0
    for (boxes, scores, labels), (truth_boxes, truth_labels) in zip(predictions,
                                                                    test_truth):
        hits_here = false_here = 0
        for class_index in range(CLASS_COUNT):
            chosen = (scores >= confidence_threshold) & (labels == class_index)
            picked_boxes, picked_scores = boxes[chosen], scores[chosen]
            if len(picked_boxes):
                keep = nms(picked_boxes, picked_scores, nms_threshold)
                picked_boxes, picked_scores = picked_boxes[keep], picked_scores[keep]
            this_truth = truth_boxes[truth_labels == class_index]
            _ordered, hits = greedy_match(picked_boxes, picked_scores, this_truth,
                                          iou_threshold)
            hits_here += int(hits.sum())
            false_here += len(picked_boxes) - int(hits.sum())
        truth_total += len(truth_boxes)
        found += hits_here
        false_total += false_here
        if hits_here == len(truth_boxes):
            full_frames += 1
    return dict(map=mean_average_precision(predictions, test_truth),
                full=full_frames / len(test_set),
                false=false_total / len(test_set),
                recall=found / truth_total)


rows = []
for name in ("чиста", "зсув 4 px", "пропуск 15 %", "клас 30 %",
             "аугм. з рамк.", "аугм. без рамок"):
    if name in MODELS:
        rows.append((name, client_metrics(MODELS[name])))

print("модель             mAP@0.5   повних кадрів   хибних на кадр   повнота")
print("-" * 72)
for name, m in rows:
    print("  %-16s %.4f       %.3f            %.2f           %.3f"
          % (name, m["map"], m["full"], m["false"], m["recall"]))

In [ ]:
def ranking(key, ascending=False):
    return [name for name, _m in sorted(rows, key=lambda r: r[1][key],
                                        reverse=not ascending)]

print("порядок за mAP:                ", " > ".join(ranking("map")))
print("порядок за повними кадрами:    ", " > ".join(ranking("full")))
print("порядок за хибними на кадр:    ", " > ".join(ranking("false", ascending=True)))
print()
best_by_false = ranking("false", ascending=True)[0]
print("Найкраща модель за «хибними на кадр» — «%s»." % best_by_false)
print("За mAP вона %d-та з %d." % (ranking("map").index(best_by_false) + 1, len(rows)))
print()
print("Ось чому одне число замовникові треба обирати з ним разом, а не за нього:")
print("метрика, яка дивиться лише на хибні спрацювання, оголошує переможцем модель,")
print("яка мовчить. Пара «повнота + хибні на кадр» такого вже не дозволить.")

## Що ми побачили

| Замір | Число |
|---|---|
| розкид від зерна на чистій розмітці | одиниця виміру для всіх порівнянь |
| зсув рамок на 4 px | найдорожча помилка розмітки в зошиті |
| пропущені рамки | `mAP` їх не бачить, а впевненість на істинних предметах падає — і купи зерен не перетинаються |
| забута строчка в аугментації | зʼїдає більшу частину виграшу від аугментації |
| випадковий поділ серій | завищує оцінку, хоча моделі однаково добрі |
| пороги | оптимум переїжджає разом із ціною помилки |
| метрика для замовника | той самий набір моделей упорядковується інакше |

Жодного разу за весь зошит ми не міняли архітектуру.

---

## Завдання

### 🟢 Рівень 1

Додай до заміру з розділу 3 ще одну настройку: **зсув рамок на 1 піксель**. Три зерна.
Порівняй падіння з розкидом від зерна й скажи, чи його взагалі впіймано.

### 🟡 Рівень 2

Побудуй теплову карту сітки порогів із розділу 8 через `plt.imshow`: по одній осі
поріг упевненості, по другій — поріг NMS, колір — вартість. Зроби дві карти для двох
різних цін помилки й познач на кожній оптимум.

### 🔴 Рівень 3

Зроби четверту політику для рамок на межі: **«обрізати, але викинути, якщо видно
менше половини»**. Реалізуй її в `apply_policy`, поміряй трьома зернами й скажи,
чи вона краща за всі три з розділу 6. Порівнюй із розкидом від зерна, а не на око.

Повний варіант цього завдання — у [домашньому](homework.html).